In [ ]:
# Phase 4 (reference data): airport coordinates + timezone, needed to
# resolve BTS's local-clock, no-timezone flight times (hubops.bts_times)
# during silver. Source: OurAirports (public-domain aviation reference
# data) -- see contracts/dim_airport.yaml for sourcing notes, including a
# real cross-source inconsistency found building this (Palm Beach Intl's
# IATA code changed to DJT in OurAirports' current data, but BTS still
# files flights under the historical PBI code).

from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import BooleanType, DoubleType

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}"

csv_path = f"{ABFSS_ROOT}/Files/airports/dim_airport.csv"
df = spark.read.option("header", True).csv(csv_path)
print(f"read {df.count():,} rows from {csv_path}")


In [ ]:
df = df.withColumn("latitude", col("latitude").cast(DoubleType()))
df = df.withColumn("longitude", col("longitude").cast(DoubleType()))
df = df.withColumn("is_aa_hub", col("is_aa_hub").cast(BooleanType()))
df = (
    df.withColumn("_source", lit("ourairports_dim_airport"))
    .withColumn("_loaded_at", current_timestamp())
)
df.printSchema()


In [ ]:
table_path = f"{ABFSS_ROOT}/Tables/dim_airport"
df.write.format("delta").mode("overwrite").save(table_path)
print(f"wrote {table_path}")


In [ ]:
check = spark.read.format("delta").load(table_path)
print(f"dim_airport row count: {check.count():,}")
check.filter(col("is_aa_hub") == True).select("iata", "name", "tz").show(truncate=False)
